In [2]:
#pip install pandas sentence-transformers faiss-cpu

In [4]:
import pandas as pd
import json
import os
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [5]:
#check working directory 

print(os.getcwd())

/Users/janajfrancis/workspace/rag-project 


In [10]:
business_file = "yelp_academic_dataset_business.json"

#check for file
if not os.path.exists(business_file):
    raise FileNotFoundError(f"File not found: {business_file}")
    
print("File found!")

File found!


In [12]:
# Load business JSON into pandas dataframe 

business_data = []

with open(business_file, "r", encoding="utf-8") as f:
    for line in f:
        business_data.append(json.loads(line))

df = pd.DataFrame(business_data)

print(f"Total businesses loaded: {len(df)}")


Total businesses loaded: 150346


In [14]:
# data exploration
print("Columns in dataset:", df.columns.tolist())

Columns in dataset: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


In [16]:
print(df.head())

              business_id                      name  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ   
1  mpf3x-BjTdTEA3yCZrAYPw             The UPS Store   
2  tUFrWirKiKi_TAnsVWINQQ                    Target   
3  MTSW4McQd7CbVtyjqoe9mw        St Honore Pastries   
4  mWMc6_wTdE0EUBKIGXDVfA  Perkiomen Valley Brewery   

                           address           city state postal_code  \
0           1616 Chapala St, Ste 2  Santa Barbara    CA       93101   
1  87 Grasso Plaza Shopping Center         Affton    MO       63123   
2             5255 E Broadway Blvd         Tucson    AZ       85711   
3                      935 Race St   Philadelphia    PA       19107   
4                    101 Walnut St     Green Lane    PA       18054   

    latitude   longitude  stars  review_count  is_open  \
0  34.426679 -119.711197    5.0             7        0   
1  38.551126  -90.335695    3.0            15        1   
2  32.223236 -110.880452    3.5            22        0   
3  39.9555

In [76]:
df['categories'].unique()

array(['Doctors, Traditional Chinese Medicine, Naturopathic/Holistic, Acupuncture, Health & Medical, Nutritionists',
       'Shipping Centers, Local Services, Notaries, Mailbox Centers, Printing Services',
       'Department Stores, Shopping, Fashion, Home & Garden, Electronics, Furniture Stores',
       ...,
       'Shopping, Jewelry, Piercing, Toy Stores, Beauty & Spas, Accessories, Fashion',
       'Fitness/Exercise Equipment, Eyewear & Opticians, Shopping, Sporting Goods, Bikes',
       'Beauty & Spas, Permanent Makeup, Piercing, Tattoo'], dtype=object)

In [74]:
df['state'].unique()

array(['CA', 'MO', 'AZ', 'PA', 'TN', 'FL', 'IN', 'LA', 'AB', 'NV', 'ID',
       'DE', 'IL', 'NJ', 'NC', 'CO', 'WA', 'HI', 'UT', 'TX', 'MT', 'MI',
       'SD', 'XMS', 'MA', 'VI', 'VT'], dtype=object)

In [78]:
filtered_df = df[
    df['categories'].str.contains("Restaurants|Food|Bars", na=False)
].copy()


In [82]:
filtered_df.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'BusinessParking': 'None', 'BusinessAcceptsCr...","Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"{'Caters': 'True', 'Alcohol': 'u'full_bar'', '...","Pubs, Restaurants, Italian, Bars, American (Tr...",None
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsAttire': ''casual'', 'Restaurants...","Ice Cream & Frozen Yogurt, Fast Food, Burgers,...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-21:0', '..."


In [18]:
# Convert all city names to lowercase and strip whitespace, then get unique values

unique_cities = df['city'].str.lower().str.strip().unique()

# print total number of unique cities 
print(f"Total unique cities: {len(unique_cities)}")

# Print the first 20 unique cities as a sample

print("Sample cities:", unique_cities[:20])



Total unique cities: 1238
Sample cities: ['santa barbara' 'affton' 'tucson' 'philadelphia' 'green lane'
 'ashland city' 'brentwood' 'st. petersburg' 'nashville' "land o' lakes"
 'tampa bay' 'indianapolis' 'clearwater' 'largo' 'new orleans' 'kenner'
 'edmonton' 'reno' 'newtown' 'white house']


In [20]:
# Filter for philadelphia

city_name = "philadelphia"

philadelphia_resturants = df[(df['city'].str.lower() == city_name) & 
(df['categories'].str.contains("Restaurants", na=False))].copy()

print(f"Number of resturants in {city_name.title()}: {len(philadelphia_resturants)}\n\n\n")

print(philadelphia_resturants[['name','categories','stars','review_count']].head())

Number of resturants in Philadelphia: 5854



                  name                                         categories  \
3   St Honore Pastries  Restaurants, Food, Bubble Tea, Coffee & Tea, B...   
15            Tuna Bar                  Sushi Bars, Restaurants, Japanese   
19                 BAP                                Korean, Restaurants   
28             Bar One  Cocktail Bars, Bars, Italian, Nightlife, Resta...   
31    DeSandro on Main                    Pizza, Restaurants, Salad, Soup   

    stars  review_count  
3     4.0            80  
15    4.0           245  
19    4.5           205  
28    4.0            65  
31    3.0            41  


In [58]:
type(philadelphia_resturants)

pandas.core.frame.DataFrame

In [60]:
philadelphia_resturants.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106,39.953949,-75.143226,4.0,245,1,"{'RestaurantsReservations': 'True', 'Restauran...","Sushi Bars, Restaurants, Japanese","{'Tuesday': '13:30-22:0', 'Wednesday': '13:30-..."
19,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147,39.943223,-75.162568,4.5,205,1,"{'NoiseLevel': 'u'quiet'', 'GoodForMeal': '{'d...","Korean, Restaurants","{'Monday': '11:30-20:30', 'Tuesday': '11:30-20..."
28,QdN72BWoyFypdGJhhI5r7g,Bar One,767 S 9th St,Philadelphia,PA,19147,39.939825,-75.157447,4.0,65,0,"{'Smoking': 'u'no'', 'NoiseLevel': 'u'average'...","Cocktail Bars, Bars, Italian, Nightlife, Resta...","{'Monday': '16:0-0:0', 'Tuesday': '16:0-0:0', ..."
31,Mjboz24M9NlBeiOJKLEd_Q,DeSandro on Main,4105 Main St,Philadelphia,PA,19127,40.022466,-75.218314,3.0,41,0,"{'RestaurantsReservations': 'False', 'Caters':...","Pizza, Restaurants, Salad, Soup","{'Tuesday': '17:0-21:30', 'Wednesday': '17:0-1..."


#### Each business is turned into a text chunk for retrieval. 
#### Chunks let the system find relevant info and generate accurate answers.

#### Each resturant becomes a searchable unit for semantic search

In [23]:
# create function to create chunks

def create_chunk(row):
    text = (
        f"{row['name']} is a {row['categories']} located at {row['address']}."
        f"It has {row['stars']} and {row['review_count']} reviews."
    ) 
    return {
        "text": text,
        "city": row['city'].title(),
        "type": "restaurant",
        "source": "Yelp business dataset"
    }
            

In [25]:
# Apply function to each row to create a list of chunks
chunks = philadelphia_resturants.apply(create_chunk, axis=1).tolist()


In [56]:
chunks[:3]

[{'text': 'St Honore Pastries is a Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries located at 935 Race St.It has 4.0 and 80 reviews.',
  'city': 'Philadelphia',
  'type': 'restaurant',
  'source': 'Yelp business dataset'},
 {'text': 'Tuna Bar is a Sushi Bars, Restaurants, Japanese located at 205 Race St.It has 4.0 and 245 reviews.',
  'city': 'Philadelphia',
  'type': 'restaurant',
  'source': 'Yelp business dataset'},
 {'text': 'BAP is a Korean, Restaurants located at 1224 South St.It has 4.5 and 205 reviews.',
  'city': 'Philadelphia',
  'type': 'restaurant',
  'source': 'Yelp business dataset'}]

In [27]:
# extract all the text from the chunks by creating a list of all texts 
texts = [chunk['text'] for chunk in chunks]

In [29]:
#texts

In [66]:
#Load the embedding model
# 'all-MiniLM-L6-v2' is a small, fast pre-trained model from SentenceTransformers.
# It converts text into a 384-dimensional vector representing semantic meaning.

#Loads a pre-trained sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [63]:
# Generate embeddings for all text chunks
# This turns each chunk's text into a vector. Similar text will have similar vectors.
# show_progress_bar=True helps visualize progress for large datasets.
embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/183 [00:00<?, ?it/s]

In [34]:
# Check the shape of embeddings
# This prints (num_chunks, embedding_dimension), e.g., (500, 384)
print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: (5854, 384)


In [35]:
#FAISS = library for fast similarity search on vectors.

#FAISS index = the storage of vectors optimized for searching.

#What it does = takes your query embedding, calculates distances to stored embeddings, and returns the most similar chunks.

In [39]:
# faiss need to know to store and compare vectors

dimension = 384  # embedding dimension from model. Each vector has 384 dimensions

# FAISS will calculate the straight-line distance between vectors when searching
# Create a vector database (FAISS index) with predefined methods for
# adding embeddings, searching for similar vectors, and inspecting stored vectors
index = faiss.IndexFlatL2(dimension)  # Use Euclidean (L2) distance

index.add(np.array(embeddings))  # add all embeddings to the index

print(f"FAISS index contains {index.ntotal} vectors.")

FAISS index contains 5854 vectors.


In [41]:
def retrieve_chunks(query, top_k=5):
    """
    Convert user query into an embedding,
    search FAISS index,
    return top-k most relevant chunks.
    """
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return results


In [43]:
query = "Highly rated pizza restaurants in Philadelphia"

results = retrieve_chunks(query, top_k=5)

for i, r in enumerate(results, start=1):
    print(f"{i}. {r['text']}\n")


1. Philly's Finest Pizza is a Restaurants, Pizza located at 7962 Dungan Rd.It has 4.5 and 23 reviews.

2. Philly Pizza and Grill is a Restaurants, Pizza located at 7315 Oxford Ave.It has 2.5 and 6 reviews.

3. Best In Town Pizza is a Restaurants, Pizza located at 7971 Castor Ave.It has 2.5 and 34 reviews.

4. Top Quality Pizza is a Pizza, Restaurants located at 6639 Castor Ave.It has 2.5 and 11 reviews.

5. City Pizza is a Restaurants, Pizza located at 100 Snyder Ave.It has 3.0 and 87 reviews.

